In [13]:
from pyflink.table import EnvironmentSettings, TableEnvironment
from faker import Faker

# Create a batch environment
env_settings = EnvironmentSettings.in_batch_mode()
table_env = TableEnvironment.create(env_settings)

# Intialize Faker
fake = Faker()

# Generate data
data = [(fake.name(), fake.city(), fake.state()) for _ in range(10)]

# Define column names
column_names = ['name', 'city', 'state']

# Create table
table = table_env.from_elements(data, schema=column_names)

# Print table
table.execute().print()

+--------------------------------+--------------------------------+--------------------------------+
|                           name |                           city |                          state |
+--------------------------------+--------------------------------+--------------------------------+
|                     Jared Long |                     Port Jason |                       Nebraska |
|               Anthony Mitchell |                      Lopeztown |                           Ohio |
|                  Michael Brown |                   Dixonchester |                    Connecticut |
|                    Erin Morton |                      Moranstad |                          Maine |
|                   Angela Ayers |                     Port Jacob |                        Montana |
|                    Allen Jones |                  South Cynthia |                     California |
|                 Dr. Troy Brown |         North Christophermouth |                        

In [21]:
table.execute().collect().next()

<Row('Jared Long', 'Port Jason', 'Nebraska')>

In [26]:
table.execute().collect().next()

<Row('Jared Long', 'Port Jason', 'Nebraska')>

In [28]:
table_env.create_temporary_view('source_table', table)

In [31]:
table_env.execute_sql(
    """
    SELECT * FROM source_table
    WHERE city = 'Lopeztown'
    """
).print()

+--------------------------------+--------------------------------+--------------------------------+
|                           name |                           city |                          state |
+--------------------------------+--------------------------------+--------------------------------+
|               Anthony Mitchell |                      Lopeztown |                           Ohio |
+--------------------------------+--------------------------------+--------------------------------+
1 row in set


### Selecting a column

In [37]:
from pyflink.table.expressions import col

table.select(col('name'), col('city')).execute().print()

+--------------------------------+--------------------------------+
|                           name |                           city |
+--------------------------------+--------------------------------+
|                     Jared Long |                     Port Jason |
|               Anthony Mitchell |                      Lopeztown |
|                  Michael Brown |                   Dixonchester |
|                    Erin Morton |                      Moranstad |
|                   Angela Ayers |                     Port Jacob |
|                    Allen Jones |                  South Cynthia |
|                 Dr. Troy Brown |         North Christophermouth |
|                   Suzanne Snow |                      Port Anna |
|                     Erik Adams |                      Laceystad |
|                 Peter Copeland |                    Kristenview |
+--------------------------------+--------------------------------+
10 rows in set


### Filtering the data

In [43]:
from pyflink.table.expressions import col

(
    table
        .filter(col('state') == 'California')
        .select(col('name'), col('city'), col('state'))
        .execute()
        .print()
)

+--------------------------------+--------------------------------+--------------------------------+
|                           name |                           city |                          state |
+--------------------------------+--------------------------------+--------------------------------+
|                    Allen Jones |                  South Cynthia |                     California |
+--------------------------------+--------------------------------+--------------------------------+
1 row in set


### GroupBy

In [59]:
from pyflink.table.expressions import col

(
    table
    .group_by(col('state'))
    .select(
        col('state').alias('state'), 
        col('name').count.alias('state_wise_count')
    )
    .execute()
    .print()
)

+--------------------------------+----------------------+
|                          state |     state_wise_count |
+--------------------------------+----------------------+
|                     California |                    1 |
|                    Connecticut |                    1 |
|                       Missouri |                    1 |
|                        Vermont |                    1 |
|                           Ohio |                    1 |
|                          Maine |                    1 |
|                        Montana |                    2 |
|                       Nebraska |                    1 |
|                         Hawaii |                    1 |
+--------------------------------+----------------------+
9 rows in set


### Creating a SINK

In [60]:
table_env.execute_sql(
    """
    CREATE TABLE print_sink (
        name STRING,
        city STRING,
        state STRING
    )
    WITH (
        'connector' = 'print'
        )
    """
)

In [63]:
table_env.execute_sql(
    """
    INSERT INTO print_sink
        SELECT * FROM source_table
    """
).wait()

2> +I[Jared Long, Port Jason, Nebraska]
2> +I[Anthony Mitchell, Lopeztown, Ohio]
2> +I[Michael Brown, Dixonchester, Connecticut]
2> +I[Erin Morton, Moranstad, Maine]
2> +I[Angela Ayers, Port Jacob, Montana]
2> +I[Allen Jones, South Cynthia, California]
2> +I[Dr. Troy Brown, North Christophermouth, Montana]
2> +I[Suzanne Snow, Port Anna, Vermont]
2> +I[Erik Adams, Laceystad, Hawaii]
2> +I[Peter Copeland, Kristenview, Missouri]


In [66]:
table_env.execute_sql(
    """
    SELECT * FROM print_sink
    """
).print()

Py4JJavaError: An error occurred while calling o490.executeSql.
: org.apache.flink.table.api.ValidationException: Unable to create a source for reading table 'default_catalog.default_database.print_sink'.

Table options are:

'connector'='print'
	at org.apache.flink.table.factories.FactoryUtil.createDynamicTableSource(FactoryUtil.java:234)
	at org.apache.flink.table.planner.plan.schema.CatalogSourceTable.createDynamicTableSource(CatalogSourceTable.java:176)
	at org.apache.flink.table.planner.plan.schema.CatalogSourceTable.toRel(CatalogSourceTable.java:116)
	at org.apache.calcite.sql2rel.SqlToRelConverter.toRel(SqlToRelConverter.java:4060)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertIdentifier(SqlToRelConverter.java:2928)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertFrom(SqlToRelConverter.java:2482)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertFrom(SqlToRelConverter.java:2393)
	at org.apache.calcite.sql2rel.SqlToRelConverter.convertFrom(SqlToRelConverte